# The Knowledge Agent (RAG)

**What you will build:** an agent that answers questions from *your* documents by retrieving the relevant passages first. This is **RAG** (retrieval-augmented generation), and it addresses two problems from 0.0: the model's knowledge is frozen, and you can't fit everything in the context window. Maps to chapter 09 of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/17_knowledge_agent_rag.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0" "pydantic-evals>=2.0,<3.0" "sentence-transformers>=3,<6"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## The idea in one picture

Don't paste all your documents into the prompt (they won't fit, and it's wasteful). Instead: **embed** each chunk into a vector, **store** the vectors, and at question time **retrieve** the few chunks closest to the question and put only those in front of the model.

```
  documents ─embed─▶ vectors ──store──┐
                                       │   question ─embed─▶ vector
                                       ▼                        │
                              similarity search  ◀──────────────┘
                                       │
                              top-k relevant chunks ─▶ into the prompt ─▶ answer
```

We use `sentence-transformers` for embeddings — it runs locally on Colab's CPU, needs **no API key**, and won't rot.

## Step 1 — Chunk, embed, store

Real knowledge doesn't arrive as tidy one-line facts — it arrives as *documents*, and the first real decision in any RAG system is how to **chunk** them: each chunk is retrieved (or missed) as a unit. Too big and one chunk matches everything vaguely; too small and answers get split across chunks that no single retrieval brings together.

Here's ACME's policy handbook as one text, and a simple chunker — split into sentences, pack them up to a size budget:

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim, CPU-friendly, no API key
# (first run downloads the model, ~90MB — one-time)

policy_text = """
ACME customer handbook. Our return policy allows items to be returned within 30 days
of purchase with a receipt. Refunds are issued to the original payment method within
5 business days of receiving the item.

ACME ships to the EU and the UK. Standard delivery takes 3 to 5 business days, and
express delivery arrives in 1 to 2 business days for an extra fee. Orders above 50
euros ship free.

The ACME Pro plan costs 20 euros per month and includes priority support and early
access to new features. The free plan covers personal use with community support.

Support hours are Monday to Friday, 9am to 6pm CET. Outside those hours, the help
center and community forum remain available.

Warranty: ACME hardware is covered for 2 years against manufacturing defects. The
warranty does not cover accidental damage or normal wear.
"""

def chunk(text: str, max_chars: int = 220) -> list[str]:
    """Split into sentences, pack them into chunks up to max_chars."""
    sentences = re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
    chunks, current = [], ""
    for s in sentences:
        if current and len(current) + len(s) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += " " + s
    if current.strip():
        chunks.append(current.strip())
    return chunks

documents = chunk(policy_text)
doc_vectors = embedder.encode(documents, normalize_embeddings=True)   # (N, 384) unit vectors

for i, d in enumerate(documents):
    print(f"[doc {i}] {d[:80]}...")
print("\nstored", len(documents), "chunks as", doc_vectors.shape, "vectors")

## Step 2 — Retrieve the most relevant chunks

Cosine similarity is just a dot product on normalized vectors. No database needed for a small set.

In [ ]:
def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_vectors @ q                    # cosine similarity to every chunk
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), documents[i]) for i in top]

for i, score, text in retrieve("how long do I have to send something back?"):
    print(f"[doc {i} | {score:.2f}] {text[:90]}")

The query said "send something back," not "return" — but retrieval still found the return-policy chunk. That's the point of embeddings: they match on **meaning**, not keywords. Keep an eye on the scores too: they're your first debugging signal (a top hit at 0.2 means "nothing really matched", long before the agent produces a bad answer).

## Step 3 — Give retrieval to the agent as a tool

The clean way to do RAG with an agent: expose retrieval as a **tool**. Now the agent decides *when* it needs to look something up — and can search more than once. This is *agentic* RAG, and it's just a tool (1.2) whose body happens to be a vector search.

In [ ]:
rag_agent = Agent(
    model,
    system_prompt=(
        "You are ACME's support assistant. Use the search_docs tool to find relevant policy "
        "before answering, and answer ONLY from what it returns — if the docs don't cover it, "
        "say you don't know. Cite the passages you used as [doc N]."
    ),
)

@rag_agent.tool_plain
def search_docs(query: str) -> str:
    """Search ACME's policy documents. Returns relevant passages tagged [doc N]."""
    return "\n".join(f"[doc {i} | {score:.2f}] {text}" for i, score, text in retrieve(query, k=2))

result = await rag_agent.run("I bought a laptop 40 days ago and it's faulty. Am I covered?")
print(result.output)

The agent searched the docs, found the 30-day return window *and* the 2-year warranty, reasoned about which applies — and **cited its sources**. Those `[doc N]` tags cost five lines and buy the number-one feature every real RAG deployment gets asked for: users can check *where* an answer came from, and you can check whether a wrong answer was a retrieval problem (wrong docs cited) or a reasoning problem (right docs, wrong conclusion). RAG is not a special framework feature; it's retrieval wired in as a tool, plus the discipline of grounding.

## Prove it works: a three-case eval

This is the main Block 1 agent, so it needs an eval. Three cases, straight from 1.6, covering the two classic RAG failures (retrieval miss and ungrounded invention):

In [ ]:
from dataclasses import dataclass
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Evaluator, EvaluatorContext

@dataclass
class AnswerContains(Evaluator[str, str]):
    def evaluate(self, ctx: EvaluatorContext[str, str]) -> bool:
        return str(ctx.expected_output).lower() in ctx.output.lower()

async def ask(question: str) -> str:
    return (await rag_agent.run(question)).output

kb_eval = Dataset(
    name="rag-smoke",
    cases=[
        Case(name="returns",   inputs="How many days do I have to return an item?", expected_output="30"),
        Case(name="pro-price", inputs="How much does the Pro plan cost per month?", expected_output="20"),
        # the grounding case: the docs say nothing about cars, so an honest agent must say it doesn't know
        Case(name="grounding", inputs="Do you sell cars?", expected_output="know"),
    ],
    evaluators=[AnswerContains()],
)

report = await kb_eval.evaluate(ask)
report.print(include_input=True, include_output=True)

Two facts and one refusal, checked automatically. The `grounding` case is the one teams forget: it asserts the agent **admits ignorance** instead of inventing policy (the substring `"know"` matches "I don't know" — crude, but it catches the failure that matters). Every fact you add to the handbook deserves a case here; that habit — grow the docs, grow the eval — is what keeps a knowledge agent trustworthy as it evolves. It's also why evals (1.6) came *before* this chapter.

::::{dropdown} 🔍 RAG vs memory vs long context
:color: info

```
Long context   paste everything      simple, but expensive and hits the window limit
Memory (0.6)   keep recent turns     good for conversation, not for a big knowledge base
RAG            fetch only what's     scales to millions of chunks; the model sees only
               relevant, on demand   the few passages it needs this turn
```

For a handful of docs, plain retrieval like this is plenty. Past ~50k chunks, swap the numpy search for `faiss-cpu` (`faiss.IndexFlatIP(384)`) — same cosine math, built for scale. In Block 2 you'll see RAG again with a grading loop (*agentic RAG*, 2.7).
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add your own document.** Replace `policy_text` with a few paragraphs about something you know, re-run the chunk/embed cell, and ask about it. **Done when:** the citations point at *your* chunks.
2. **Watch grounding work.** Ask something the docs don't cover ("do you sell cars?") and confirm it says it doesn't know instead of inventing. Then delete the "say you don't know" line from the system prompt and re-run. **Done when:** you've seen the invention happen — that's the failure the line prevents.
3. **Tune the two knobs.** Try `k=1` vs `k=4`, and `max_chars=80` vs `max_chars=500`, re-running the eval each time. **Done when:** you can describe how each knob failed at its extreme.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**2 —** Without the grounding instruction, most models answer car questions helpfully and fluently — pure invention, delivered with confidence. The eval's `grounding` case goes red the moment you remove the line, which is exactly what an eval is for: it remembers the failure so you don't have to.

**3 — What the extremes look like:** `k=1` misses multi-chunk answers (the laptop question needs returns *and* warranty — with one chunk the agent sees only half the picture). `k=4` on a five-chunk corpus retrieves nearly everything, so "retrieval" stops filtering — fine here, fatal at scale (irrelevant chunks crowd the prompt and the model quotes the wrong one). `max_chars=80` splits the refund rule mid-sentence across chunks that never co-retrieve; `max_chars=500` merges returns+shipping into one blob that matches everything a little and nothing well. There is no universal setting — there is your corpus, your questions, and the eval that tells you when you've got it right.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **Says "I don't know" when the answer IS in the docs** | Retrieval missed the chunk (`k` too small, chunks too big) | Raise `k`; split documents into smaller chunks |
| **Retrieves irrelevant passages** | Query and docs phrased very differently | Try a stronger embedding model; add a query-rewrite step (see 2.7) |
| **Answers from its own knowledge, not the docs** | System prompt not strict enough | Tell it to answer ONLY from retrieved passages, else say "I don't know" |
| **Slow on many documents** | Linear numpy scan over all vectors | Switch to `faiss-cpu` (`IndexFlatIP`) for large collections |
::::

You can now build clean, typed, tool-using agents with memory, guardrails, evals, and retrieval — the core toolkit for most real applications.

**What's next:** two more Block 1 skills — in **18** you get tools from an **MCP server** (capabilities you didn't write), and in **19** you learn to **debug** an agent when it misbehaves.